# Projet Machine Learning — Prediction du Diabete




In [ ]:
# installer les dependances si elles ne sont pas deja presentes
# executer cette cellule une seule fois avant le reste du notebook
!pip install numpy pandas matplotlib seaborn scikit-learn -q

: 

In [ ]:
# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             roc_curve)

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
np.random.seed(42)
print("librairies chargees")

## 2. Chargement et analyse exploratoire (EDA)


In [ ]:
# chargement depuis le fichier local
df = pd.read_csv("diabetes.csv")

print("Dimensions :", df.shape)
df.head(5)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# distribution de la variable cible
print(df['Outcome'].value_counts())
print("\nProportion :")
print(df['Outcome'].value_counts(normalize=True).round(3))

sns.countplot(x='Outcome', data=df, palette='Set2')
plt.title("Repartition des classes (0 = sain, 1 = diabetique)")
plt.show()

In [ ]:
# matrice de correlation
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Matrice de correlation")
plt.show()

In [ ]:
# comptage des 0 impossibles
cols_zero_impossibles = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print("Nombre de 0 impossibles par colonne :")
print((df[cols_zero_impossibles] == 0).sum())

## 4. Pretraitement et Feature Engineering

Etapes :
1. Remplacer les zeros impossibles par `NaN`.
2. Imputer par la mediane conditionnee a la classe (plus pertinent qu'une mediane globale).
3. Creer quelques nouvelles variables : categories d'IMC, d'age, de glycemie.
4. Standardiser toutes les variables.


In [ ]:
# remplacer les 0 impossibles par NaN
df_clean = df.copy()
df_clean[cols_zero_impossibles] = df_clean[cols_zero_impossibles].replace(0, np.nan)
print("Valeurs manquantes apres remplacement :")
print(df_clean.isnull().sum())

In [ ]:
# imputation par la mediane conditionnee a la classe (Outcome)
for col in cols_zero_impossibles:
    df_clean[col] = df_clean.groupby('Outcome')[col].transform(lambda x: x.fillna(x.median()))

print("Valeurs manquantes apres imputation :", df_clean.isnull().sum().sum())

In [ ]:
# feature engineering : variables categoriales derivees
df_clean['BMI_cat'] = pd.cut(df_clean['BMI'],
                             bins=[0, 18.5, 25, 30, 100],
                             labels=[0, 1, 2, 3]).astype(int)  # maigre/normal/surpoids/obese

df_clean['Age_cat'] = pd.cut(df_clean['Age'],
                             bins=[20, 30, 45, 100],
                             labels=[0, 1, 2]).astype(int)      # jeune/moyen/age

df_clean['Glucose_cat'] = pd.cut(df_clean['Glucose'],
                                 bins=[0, 100, 125, 300],
                                 labels=[0, 1, 2]).astype(int)  # normal/pre-diabete/diabete

df_clean.head()

In [ ]:
# separation features / cible + split train/test stratifie
X = df_clean.drop('Outcome', axis=1)
y = df_clean['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# standardisation (fit sur train uniquement pour eviter le data leakage)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print("Train :", X_train_s.shape, " | Test :", X_test_s.shape)

## 5. Entrainement et comparaison des modeles

On teste 5 algorithmes de classification classiques et on les compare sur plusieurs metriques. Vu le desequilibre des classes, on regarde principalement le F1-score et l'AUC.


In [ ]:
models = {
    'Regression Logistique': LogisticRegression(max_iter=1000, random_state=42),
    'KNN':                   KNeighborsClassifier(),
    'SVM':                   SVC(probability=True, random_state=42),
    'Random Forest':         RandomForestClassifier(random_state=42),
    'Gradient Boosting':     GradientBoostingClassifier(random_state=42),
}

resultats = []
for nom, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred  = model.predict(X_test_s)
    y_proba = model.predict_proba(X_test_s)[:, 1]
    resultats.append({
        'Modele':    nom,
        'Accuracy':  round(accuracy_score(y_test, y_pred), 3),
        'Precision': round(precision_score(y_test, y_pred), 3),
        'Recall':    round(recall_score(y_test, y_pred), 3),
        'F1-score':  round(f1_score(y_test, y_pred), 3),
        'AUC':       round(roc_auc_score(y_test, y_proba), 3),
    })

df_resultats = pd.DataFrame(resultats).sort_values('F1-score', ascending=False).reset_index(drop=True)
df_resultats

In [ ]:
# comparaison visuelle des modeles
df_plot = df_resultats.set_index('Modele')[['Accuracy', 'F1-score', 'AUC']]
df_plot.plot(kind='bar', figsize=(11, 5), colormap='viridis')
plt.title("Comparaison des modeles")
plt.ylabel("Score")
plt.xticks(rotation=20, ha='right')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 6. Optimisation du meilleur modele (GridSearch)

On cherche les meilleurs hyperparametres du Gradient Boosting via une validation croisee a 5 folds.


In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [2, 3, 4],
}

grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train_s, y_train)

print("Meilleurs hyperparametres :", grid.best_params_)
print("Meilleur F1 (validation croisee) :", round(grid.best_score_, 3))

best_model = grid.best_estimator_

## 7. Evaluation finale du modele optimise


In [ ]:
y_pred  = best_model.predict(X_test_s)
y_proba = best_model.predict_proba(X_test_s)[:, 1]

print("=== Rapport de classification ===")
print(classification_report(y_test, y_pred, target_names=['Sain (0)', 'Diabetique (1)']))

In [ ]:
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)

print(f"Accuracy  : {accuracy:.3f}")
print(f"Precision : {precision:.3f}")
print(f"Recall    : {recall:.3f}")
print(f"F1-score  : {f1:.3f}")

In [ ]:
# matrice de confusion
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Sain', 'Diabetique'],
            yticklabels=['Sain', 'Diabetique'])
plt.xlabel("Prediction"); plt.ylabel("Realite")
plt.title("Matrice de confusion")
plt.show()

In [ ]:
# courbe ROC
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
plt.plot(fpr, tpr, label=f'Gradient Boosting (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Aleatoire')
plt.xlabel("Taux de faux positifs"); plt.ylabel("Taux de vrais positifs")
plt.title("Courbe ROC")
plt.legend()
plt.show()

In [ ]:
# importance des variables
importances = pd.Series(best_model.feature_importances_, index=X.columns).sort_values()
importances.plot(kind='barh', figsize=(8, 6), color='teal')
plt.title("Importance des variables")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
# --- prediction interactive ---
# remplir chaque champ et appuyer sur Entree

Pregnancies              = int(input("Nombre de grossesses         : "))
Glucose                  = float(input("Glycemie (mg/dL)            : "))
BloodPressure            = float(input("Pression arterielle (mm Hg) : "))
SkinThickness            = float(input("Epaisseur pli cutane (mm)   : "))
Insulin                  = float(input("Insuline (mu U/ml)          : "))
BMI                      = float(input("IMC (kg/m2)                 : "))
DiabetesPedigreeFunction = float(input("Score heredite diabete       : "))
Age                      = int(input("Age (annees)                : "))

# calcul automatique des categories derivees
BMI_cat     = int(np.digitize(BMI,     [18.5, 25, 30]))
Age_cat     = int(np.digitize(Age,     [30, 45]))
Glucose_cat = int(np.digitize(Glucose, [100, 125]))

# assembler et predire
exemple = pd.DataFrame(
    [[Pregnancies, Glucose, BloodPressure, SkinThickness,
      Insulin, BMI, DiabetesPedigreeFunction, Age,
      BMI_cat, Age_cat, Glucose_cat]],
    columns=X.columns
)

exemple_s   = scaler.transform(exemple)
prediction  = best_model.predict(exemple_s)[0]
probabilite = best_model.predict_proba(exemple_s)[0][1]

resultat = "Diabetique" if prediction == 1 else "Non diabetique"
print(f"  Resultat    : {resultat}")
print(f"  Probabilite : {probabilite:.1%}")

---

## 9. Problemes rencontres et resolutions

Pour chaque erreur classique on montre : le code qui pose probleme, puis la correction appliquee dans ce projet.


### Probleme 1 — Data leakage avec le StandardScaler *(Coding)*

On scale tout le dataset avant le split. Le scaler "voit" les donnees de test pendant son fit, ce qui donne des resultats trop optimistes en evaluation.


In [ ]:
# MAUVAIS — data leakage
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

scaler_bad = StandardScaler()
X_scaled_bad = scaler_bad.fit_transform(X)            # le test est inclus !
X_tr_bad, X_te_bad, y_tr_bad, y_te_bad = train_test_split(
    X_scaled_bad, y, test_size=0.2, stratify=y, random_state=42)

lr_bad = LogisticRegression(max_iter=1000, random_state=42)
lr_bad.fit(X_tr_bad, y_tr_bad)
f1_bad = f1_score(y_te_bad, lr_bad.predict(X_te_bad))
print(f"F1 avec data leakage   : {f1_bad:.3f}  <- score gonfle artificiellement")

In [ ]:
# CORRECT — fit du scaler sur train uniquement
X_tr_ok, X_te_ok, y_tr_ok, y_te_ok = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

scaler_ok = StandardScaler()
X_tr_ok_s = scaler_ok.fit_transform(X_tr_ok)   # fit sur train seulement
X_te_ok_s = scaler_ok.transform(X_te_ok)       # transform sans refit

lr_ok = LogisticRegression(max_iter=1000, random_state=42)
lr_ok.fit(X_tr_ok_s, y_tr_ok)
f1_ok = f1_score(y_te_ok, lr_ok.predict(X_te_ok_s))
print(f"F1 sans data leakage   : {f1_ok:.3f}  <- estimation reelle")
print(f"Difference             : {f1_bad - f1_ok:+.3f}")

### Probleme 3 — Garder les zeros impossibles sans les traiter *(Feature / Data)*

Glucose = 0, BMI = 0 sont physiologiquement impossibles. Si on les garde, le modele apprend sur des donnees fausses et ses predictions sont biaisees.


In [ ]:
# MAUVAIS — entrainer directement sur le dataset brut avec les zeros
from sklearn.ensemble import GradientBoostingClassifier

X_raw = df.drop('Outcome', axis=1)
y_raw = df['Outcome']
X_r_tr, X_r_te, y_r_tr, y_r_te = train_test_split(
    X_raw, y_raw, test_size=0.2, stratify=y_raw, random_state=42)

sc_raw = StandardScaler()
X_r_tr_s = sc_raw.fit_transform(X_r_tr)
X_r_te_s = sc_raw.transform(X_r_te)

gb_raw = GradientBoostingClassifier(random_state=42)
gb_raw.fit(X_r_tr_s, y_r_tr)
f1_raw = f1_score(y_r_te, gb_raw.predict(X_r_te_s))
print(f"F1 avec zeros bruts (donnees fausses) : {f1_raw:.3f}")

In [ ]:
# CORRECT — apres remplacement des zeros par NaN et imputation par mediane
print(f"F1 apres nettoyage et imputation       : {f1_score(y_test, y_pred):.3f}")
print()
print("Zeros avant traitement :")
print((df[['Glucose','BloodPressure','SkinThickness','Insulin','BMI']] == 0).sum())
print()
print("Zeros apres traitement :")
print((df_clean[['Glucose','BloodPressure','SkinThickness','Insulin','BMI']] == 0).sum())

### Probleme 4 — Oublier `stratify=y` dans le split *(Coding)*

Sans stratify, la proportion de diabetiques dans train et test peut varier de facon aleatoire. Sur un petit dataset comme celui-ci, l'ecart peut etre significatif et fausser l'evaluation.


In [ ]:
# MAUVAIS — sans stratify
_, _, _, y_te_no = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Sans stratify — proportion diabetiques dans test : {y_te_no.mean():.3f}")

In [ ]:
# CORRECT — avec stratify=y
_, _, _, y_te_st = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Avec stratify  — proportion diabetiques dans test : {y_te_st.mean():.3f}")
print(f"Proportion dans le dataset complet                : {y.mean():.3f}")
print()
print("-> stratify garantit que train et test ont la meme distribution de classes.")

### Probleme 5 — Ne pas normaliser avant KNN et SVM *(Feature Engineering)*

KNN et SVM calculent des distances. Sans normalisation, Insulin (0–846) ecrase completement DiabetesPedigreeFunction (0–2.4). Le modele devient insensible aux features a petite echelle.


In [ ]:
# MAUVAIS — KNN sans normalisation
from sklearn.neighbors import KNeighborsClassifier

knn_raw = KNeighborsClassifier()
X_r2_tr, X_r2_te, y_r2_tr, y_r2_te = train_test_split(
    X_raw, y_raw, test_size=0.2, stratify=y_raw, random_state=42)
knn_raw.fit(X_r2_tr, y_r2_tr)
f1_knn_raw = f1_score(y_r2_te, knn_raw.predict(X_r2_te))
print(f"KNN sans normalisation : F1 = {f1_knn_raw:.3f}")
print()
print("Ecart-types des features brutes (avant normalisation) :")
print(X_raw.std().sort_values(ascending=False).round(1))

In [ ]:
# CORRECT — KNN avec normalisation
knn_ok = KNeighborsClassifier()
knn_ok.fit(X_train_s, y_train)
f1_knn_ok = f1_score(y_test, knn_ok.predict(X_test_s))
print(f"KNN avec normalisation : F1 = {f1_knn_ok:.3f}")
print(f"Gain                   : {f1_knn_ok - f1_knn_raw:+.3f}")
print()
print("Ecart-types apres StandardScaler (tout ramene autour de 1) :")
import pandas as pd
print(pd.DataFrame(X_train_s, columns=X.columns).std().sort_values(ascending=False).round(3))

### Probleme 6 — Overfitting : bon score train, mauvais score test *(Modeling)*

Un Random Forest sans contrainte memorise les donnees d'entrainement. Le score train est excellent mais le modele generalise mal. On le detecte en comparant train score et test score, et en utilisant la validation croisee.


In [ ]:
# MAUVAIS — Random Forest sans contrainte de profondeur
rf_libre = RandomForestClassifier(random_state=42)
rf_libre.fit(X_train_s, y_train)

f1_train = f1_score(y_train, rf_libre.predict(X_train_s))
f1_test  = f1_score(y_test,  rf_libre.predict(X_test_s))
print(f"Random Forest sans contrainte")
print(f"  F1 train : {f1_train:.3f}  <- memorise le train")
print(f"  F1 test  : {f1_test:.3f}  <- generalise moins bien")
print(f"  Overfitting (ecart) : {f1_train - f1_test:.3f}")

In [ ]:
# CORRECT — Random Forest avec contrainte + validation croisee pour detecter l'overfitting
rf_contraint = RandomForestClassifier(max_depth=5, n_estimators=100, random_state=42)
rf_contraint.fit(X_train_s, y_train)

f1_tr_c = f1_score(y_train, rf_contraint.predict(X_train_s))
f1_te_c = f1_score(y_test,  rf_contraint.predict(X_test_s))
print(f"Random Forest avec max_depth=5")
print(f"  F1 train : {f1_tr_c:.3f}")
print(f"  F1 test  : {f1_te_c:.3f}")
print(f"  Overfitting (ecart) : {f1_tr_c - f1_te_c:.3f}  <- beaucoup reduit")
print()
cv = cross_val_score(rf_contraint, X_train_s, y_train, cv=5, scoring='f1')
print(f"  Validation croisee (5 folds) : {cv.mean():.3f} +/- {cv.std():.3f}  <- estimation fiable")